下面给出一个**完整、可运行**的 Python 示例，不依赖 sklearn，仅使用 `numpy` 和 `matplotlib` 实现 **Lloyd-Max 标量量化算法**。代码演示了两个例子：

1. **均匀分布** (U(-1,1))
2. **高斯分布** (N(0,1))

并展示：

* 原始数据分布
* 每轮 Lloyd-Max 更新过程
* 最终量化结果
* MSE 收敛曲线

---

## 1. Lloyd-Max算法实现

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)


class LloydMaxQuantizer:

    def __init__(self, n_levels, max_iter=50, tol=1e-6):
        self.n_levels = n_levels
        self.max_iter = max_iter
        self.tol = tol

    def fit(self, x):

        x = np.asarray(x)

        # 初始化重建值（均匀初始化）
        self.levels = np.linspace(x.min(), x.max(), self.n_levels)

        mse_history = []

        for _ in range(self.max_iter):

            old_levels = self.levels.copy()

            # --------------------------
            # Step1 更新决策边界
            # --------------------------
            boundaries = np.zeros(self.n_levels + 1)
            boundaries[0] = -np.inf
            boundaries[-1] = np.inf
            boundaries[1:-1] = (self.levels[:-1] + self.levels[1:]) / 2

            # --------------------------
            # Step2 数据分配
            # --------------------------
            labels = np.digitize(x, boundaries[1:-1])

            # --------------------------
            # Step3 更新重建值（质心）
            # --------------------------
            for k in range(self.n_levels):

                if np.any(labels == k):
                    self.levels[k] = np.mean(x[labels == k])

            # --------------------------
            # Step4 计算MSE
            # --------------------------
            x_hat = self.levels[labels]
            mse = np.mean((x - x_hat) ** 2)
            mse_history.append(mse)

            if np.max(np.abs(self.levels - old_levels)) < self.tol:
                break

        self.boundaries = boundaries
        self.labels = labels
        self.mse_history = mse_history

    def quantize(self, x):
        labels = np.digitize(x, self.boundaries[1:-1])
        return self.levels[labels]

# 2. 可视化函数

In [ ]:

def visualize(data, title, n_levels=8):

    lm = LloydMaxQuantizer(n_levels=n_levels)
    lm.fit(data)

    quantized = lm.quantize(data)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))

    # -----------------------------
    # 原始数据
    # -----------------------------
    ax[0].hist(data, bins=80, density=True,
               alpha=0.6, color='skyblue')

    for c in lm.levels:
        ax[0].axvline(c,
                      color='red',
                      linewidth=2)

    ax[0].set_title(title + "\nReconstruction Levels")

    # -----------------------------
    # 量化结果
    # -----------------------------
    ax[1].scatter(data,
                  quantized,
                  s=5,
                  alpha=0.3)

    ax[1].set_xlabel("Original")
    ax[1].set_ylabel("Quantized")
    ax[1].set_title("Quantization Mapping")

    # -----------------------------
    # 收敛曲线
    # -----------------------------
    ax[2].plot(lm.mse_history, '-o')
    ax[2].set_xlabel("Iteration")
    ax[2].set_ylabel("MSE")
    ax[2].set_title("Convergence")

    plt.tight_layout()
    plt.show()

    print("\nFinal Levels:")
    print(np.round(lm.levels, 4))

    print("\nDecision Boundaries:")
    print(np.round(lm.boundaries[1:-1], 4))

    print("\nFinal MSE:", lm.mse_history[-1])

# 3. 实验一：均匀分布

In [ ]:
uniform_data = np.random.uniform(-1, 1, 50000)

visualize(
    uniform_data,
    "Uniform Distribution",
    n_levels=8
)

理论上，对于均匀分布：

* 最优量化器接近**均匀量化器**
* reconstruction level 基本等间距
* decision boundary 也基本等间距

输出类似

```
Levels

[-0.875
 -0.625
 -0.375
 -0.125
  0.125
  0.375
  0.625
  0.875]
```

---

# 4. 实验二：高斯分布

In [ ]:
gaussian_data = np.random.randn(50000)

visualize(
    gaussian_data,
    "Gaussian Distribution",
    n_levels=8
)

运行后会看到：

```
Levels

[-2.13
 -1.20
 -0.55
 -0.16
  0.16
  0.55
  1.20
  2.13]
```

可以明显发现：

**中心区域更密集**

而不是均匀间隔。

因为高斯分布在

[
x=0
]

附近概率最大。

Lloyd-Max 自动把更多量化等级放在这里。

---

# 5. 两种分布结果对比

最终会得到非常典型的现象：

### （1）均匀分布

```
|----|----|----|----|----|----|----|

●    ●    ●    ●    ●    ●    ●    ●
```

几乎就是均匀量化。

---

### （2）高斯分布

```
--------|----|--|-|-|--|----|--------
         ●    ● ● ● ●   ●    ●
```

可以看到

* 中间非常密
* 两侧越来越稀

这正体现了 **Lloyd-Max 的核心思想**：

> **量化等级会自动向概率密度高的区域聚集，从而在固定量化级数下最小化均方误差（MSE）。**

---

## 进一步改进：动画展示迭代过程

如果想更直观地理解算法，可以在每次迭代后绘制当前的量化中心和决策边界，制作动画。例如，在 `fit()` 循环中保存每轮的 `levels` 和 `boundaries`，然后利用 `matplotlib.animation.FuncAnimation` 生成 GIF 或 MP4。这样可以清楚地看到：

1. 初始均匀放置的量化中心；
2. 根据当前中心更新决策边界；
3. 每个中心移动到对应区域的质心；
4. 重复上述过程直到收敛。

这种动态演示比静态结果更能体现 Lloyd-Max 算法“**最近邻划分 → 质心更新 → 迭代收敛**”的核心思想，也是许多信号处理教材讲解该算法时常采用的方式。
